# Technical Methodology

## Objective

The technical analysis asked whether read cleaning changes downstream gene-count profiles enough to justify applying cleaning before biological analysis. Each SRR was processed under multiple cleaning strategies and then compared at the count-profile level.

## Data Selection

The initial processing queue contained 55 BioProjects and 1247 SRR accessions. This queue was intentionally broader than the final DESeq2-ready panel because it included candidate projects that still had to pass download, conversion, platform, library-strategy, and metadata checks. Of the 1247 queued SRRs, 1201 reached the final `done` state. The remaining 46 SRRs were not used for the main technical comparisons: 36 were marked `abandoned`, 9 were marked `aborted`, and 1 remained in a fetch-failed state.

| Selection or processing step | BioProjects | SRRs | Notes |
|---|---:|---:|---|
| Initial acquisition queue | 55 | 1247 | Candidate SRRs collected from BioProject/SRA metadata before technical filtering |
| Completed queue entries | 54 | 1201 | SRRs with completed local processing status (`done`) |
| Not completed | - | 46 | 36 abandoned, 9 SLURM-aborted, 1 fetch-failed |
| Platform/library evaluated | - | 1191 | SRRs with platform and library metadata available for filtering |
| Illumina retained | - | 1155 | Non-Illumina runs were archived outside the active technical analysis |
| Non-Illumina archived | - | 36 | Oxford Nanopore and BGISEQ runs in the candidate pool |
| Illumina RNA-seq, broad flag | - | 917 | `is_illumina=yes` and `is_rnaseq=yes` in platform-filter table |
| Illumina `RNA-Seq` strategy, strict panel | 48 | 892 | Curated sample sheets used for downstream DESeq2 project review |
| 2-4 class cap for biological DESeq2 pass | 28 | 485 | Projects retained under the current class cap |

The non-completed SRRs were excluded for practical technical reasons rather than biological interpretation. The queue status fields distinguish broad failure modes: 11 SRRs were marked `FETCH_ABANDONED`, 5 had incomplete count outputs after processing (`COUNT_OUTPUTS_INCOMPLETE_ABANDONED`), 20 belonged to a project abandoned because too few usable results remained (`PROJECT_ABANDONED_LT70_RESULTS`), 9 were marked `SLURM_ABORTED`, and 1 retained a `FETCH_FAILED` state. These labels cover the cases where files were too large or costly to keep retrying within the available allocation, could not be retrieved reliably, failed during SRA-to-FASTQ conversion/decompression, or produced incomplete downstream count outputs. Because the database stores the final queue state rather than a detailed causal audit trail for every retry, these categories are reported as operational filtering classes rather than as mutually exclusive biological exclusions.

SRRs were then checked for sequencing platform and library strategy. Non-Illumina runs were separated from the active analysis set because the technical comparison was designed around Illumina short-read RNA-seq processing. In total, 1191 SRRs had platform/library metadata evaluated; 1155 Illumina SRRs were retained at this stage and 36 non-Illumina SRRs were marked for archive. The broad RNA-seq flag retained 917 Illumina RNA-seq-like SRRs, including transcriptomic single-cell records. For the curated DESeq2 sample sheets, we used the stricter Illumina `RNA-Seq` strategy panel, yielding 48 BioProjects and 892 SRRs. Applying the current cap of 2-4 classes leaves 28 BioProjects and 485 SRRs for the planned biological DESeq2/GSEA pass.

SRR accessions were selected from the BioProject/SRA metadata tables, and sequencing layout, platform, instrument model, library strategy, library source, and submitted metadata were retained for filtering and interpretation. SRA objects were retrieved with SRA Toolkit `prefetch`, converted to FASTQ with `fasterq-dump`, and compressed before processing. Paired-end and single-end libraries were processed with their corresponding aligner and counting modes, and the declared Phred encoding was passed through to Trimmomatic and Bowtie2. The technical comparisons were made within SRR, so each cleaning branch for a sample used the same raw input, layout, reference, annotation, aligner, and counter.

Samples were included in the main technical summaries when the corresponding count files were available for the compared modes. Missing outputs from failed or overly aggressive branches were treated as unavailable for that specific comparison rather than being imputed. Non-Illumina runs were archived outside the active technical analysis. P35 was retained in the technical output as an aggressive stress-test condition, but analyses were also summarized without P35 because it is not intended as a routine cleaning strategy.


## BioProject Panel

This table summarizes the curated BioProject panel that will be used to decide which projects proceed to the biological DESeq2/GSEA analysis. The technical analysis itself was broader, but the class cap below marks the projects currently retained for the next biological pass.

| BioProject | SRRs | Classes | Capped at 2-4 classes | Metadata status | Condition counts |
|---|---:|---:|---|---|---|
| PRJNA1014106 | 4 | 1 | no | manual_one_class | ataxia_pbmc:4 |
| PRJNA1014965 | 37 | 2 | yes | manual_two_class | normal:20; tumor:17 |
| PRJNA1105191 | 33 | 11 | no | manual_multi_class | dms53:3; dms53_sictrl:3; dms53_sinfic:3; h1417:3; h69:3; h841:3; h841_sictrl:3; h841_sinfic:3; ipsc_c6:3; ipsc_c7:3; saec:3 |
| PRJNA1120369 | 38 | 3 | yes | manual_multi_class | mega_01:14; mega_02:16; wt:8 |
| PRJNA1127555 | 12 | 3 | yes | manual_multi_class | engager:4; naive:4; tumor:4 |
| PRJNA1133701 | 36 | 12 | no | manual_multi_class | tp53ko_control:3; tp53ko_d_e:3; tp53ko_decitabine:3; tp53ko_etoposide:3; tp53rescue_control:3; tp53rescue_d_e:3; tp53rescue_decitabine:3;... |
| PRJNA1165739 | 40 | 4 | yes | manual_multi_class | hl60_ccl240:10; k562_ccl243:10; kasumi1_crl2724:10; thp1_tib202:10 |
| PRJNA1175639 | 16 | 4 | yes | manual_multi_class | brown_differentiated:4; brown_preadipocyte_d0:4; white_differentiated:4; white_preadipocyte_d0:4 |
| PRJNA1176539 | 37 | 13 | no | manual_multi_class | cell_culture_medium:1; mrc5_cells:1; ntc_pbs:1; raji_cells:2; raji_ramos_ratio_0p00001:1; raji_ramos_ratio_0p00005:1; raji_ramos_ratio_0p... |
| PRJNA1185243 | 8 | 5 | no | manual_multi_class | wu382_65_d57_lymph_node:1; wu382_67_d57_lymph_node:2; wu382_69_d57_lymph_node:2; wu382_70_d57_lymph_node:2; wu382_71_d57_lymph_node:1 |
| PRJNA316201 | 8 | 1 | no | manual_one_class | t47d_breast_cancer_cell_line:8 |
| PRJNA321028 | 16 | 3 | yes | manual_multi_class | stage_i_endometrial_cancer:4; stage_ii_endometrial_cancer:9; stage_iii_endometrial_cancer:3 |
| PRJNA321087 | 19 | 4 | yes | manual_multi_class | cancer:8; epithelium:5; gland:2; muscle:4 |
| PRJNA321967 | 12 | 4 | yes | manual_multi_class | primary_spermatocytes:3; sertoli_cells:3; spermatids:3; undifferentiated_spermatogonia:3 |
| PRJNA352875 | 12 | 4 | yes | manual_multi_class | control:3; rho0:3; rho0_tnf:3; tnf_treated_control:3 |
| PRJNA373978 | 15 | 2 | yes | two_class_candidate | leptotene_zygotene_spermatocytes:5; pachytene_spermatocytes:10 |
| PRJNA378952 | 4 | 2 | yes | manual_two_class | lsc:1; wdlps:3 |
| PRJNA381115 | 18 | 18 | no | manual_multi_class | bel_7402:1; csqt_1:1; csqt_2:1; hcclm1_s11:1; hcclm1_s3:1; hcclm1_s4:1; hcclm1_s5:1; hcclm3:1; hcclnm1_s11:1; hcclnm1_s13:1; hep3b:1; hep... |
| PRJNA381757 | 13 | 3 | yes | manual_multi_class | fibroblast:3; naive_induced_pluripotent_stem_cells:9; primed_pluripotent_stem_cells:1 |
| PRJNA384289 | 14 | 2 | yes | manual_two_class | circulating_tumour_cells:9; primary_colorectal_tumour:5 |
| PRJNA386992 | 21 | 2 | yes | manual_two_class | aml_case:19; remission:2 |
| PRJNA397941 | 43 | 15 | no | manual_multi_class | cnhsm:1; crset:2; ct2ilgoy_k:3; ct2ilgoy_oskm:3; e8_primed:5; e8_primed_d2_no_klf4:2; e8_primed_d2_with_klf4:2; e8_primed_d6_no_klf4:2; e... |
| PRJNA416439 | 48 | 2 | yes | manual_two_class | adjacent_normal_liver_tissue:24; hepatoblastoma:24 |
| PRJNA419934 | 21 | 14 | no | manual_multi_class | parental_caov3:2; r06_step15:2; r06_step5:1; r14_step10:1; r14_step15:2; r14_step15a:1; r14_step5:1; r16_step15:2; r16_step5:1; r18_step1... |
| PRJNA451395 | 10 | 3 | yes | manual_multi_class | control:3; smc1a_c_a2027g:4; smc1a_wild_type:3 |
| PRJNA479479 | 12 | 4 | yes | manual_multi_class | p1_scm:3; p1_tcp:3; p4_scm:3; p4_tcp:3 |
| PRJNA480287 | 1 | 1 | no | manual_not_for_deseq2 | not_for_deseq2:1 |
| PRJNA509121 | 39 | 6 | no | manual_multi_class | older_5h_post_concentric:7; older_5h_post_eccentric:6; older_baseline:7; young_5h_post_concentric:6; young_5h_post_eccentric:6; young_bas... |
| PRJNA528522 | 19 | 6 | no | manual_multi_class | hepatic_endoderm:5; mature_hepatocytes:5; organoid_differentiation_medium:3; organoid_expansion_medium:1; organoid_hepatic_medium:1; plur... |
| PRJNA544334 | 28 | 15 | no | manual_multi_class | bone_marrow_cd133p_hsc:1; bone_marrow_cd34p_hspc:2; bone_marrow_cd38low_hsc:1; bone_marrow_cd90p_hsc:2; bone_marrow_emp:2; bone_marrow_hs... |
| PRJNA610985 | 9 | 3 | yes | manual_multi_class | siartd1_knockdown_doxycycline:3; simock_doxycycline:3; simock_untreated:3 |
| PRJNA623750 | 12 | 4 | yes | manual_multi_class | a549_treated:3; a549_untreated:3; hct116_treated:3; hct116_untreated:3 |
| PRJNA625628 | 11 | 2 | yes | manual_two_class | fb:3; lcl:8 |
| PRJNA634200 | 30 | 4 | yes | manual_multi_class | desmoplastic_normal:9; desmoplastic_tumor:9; replacement_normal:6; replacement_tumor:6 |
| PRJNA664293 | 12 | 6 | no | manual_multi_class | egfp_cag27:2; egfp_cag27_db213:2; egfp_cag78:2; egfp_cag78_db213:2; untransfected:2; untransfected_db213:2 |
| PRJNA673295 | 12 | 4 | yes | manual_multi_class | ame:3; ame_tio2:3; control:3; tio2:3 |
| PRJNA673745 | 12 | 4 | yes | manual_multi_class | control:3; egcg:3; zno2:3; zno2_egcg:3 |
| PRJNA734133 | 7 | 2 | yes | manual_two_class | caki1_kaiso_ko:4; caki1_wt:3 |
| PRJNA752868 | 20 | 2 | yes | manual_two_class | cd4_mature_naive:10; naive_treg:10 |
| PRJNA767228 | 15 | 5 | no | manual_multi_class | mcf102a:3; mcf7:3; mdamb231:3; mdamb361:3; skbr3:3 |
| PRJNA786266 | 15 | 3 | yes | manual_multi_class | aeh:5; eec:5; normal_endometrium:5 |
| PRJNA788948 | 19 | 14 | no | manual_multi_class | cd14_monocytes:1; cd19_b_cells:1; cd34_normal_donor:1; cd34_patient_gt39m:1; cd3_t_cells:1; ipsc_clone_0copy:2; ipsc_clone_10copy:2; ipsc... |
| PRJNA838478 | 12 | 4 | yes | manual_multi_class | control_cell:1; hydronephrosis_patient_cell:1; hydronephrosis_patient_supernatant:2; unlabeled_amniotic_fluid:8 |
| PRJNA876028 | 6 | 2 | yes | manual_two_class | kasumi1_control:3; kasumi1_fto_overexpression:3 |
| PRJNA903521 | 11 | 6 | no | manual_multi_class | ipf_fibrosis_clone_ali:2; ipf_fibrosis_clone_sc:2; ipf_normal_clone_ali:2; ipf_normal_clone_sc:2; scrna_control_lung_stem_pool:1; scrna_i... |
| PRJNA976462 | 32 | 9 | no | manual_multi_class | pt1:4; pt13:4; pt15:4; pt2:4; pt3:2; pt4:2; pt5:4; pt7:4; pt9:4 |
| PRJNA987832 | 12 | 6 | no | manual_multi_class | 293ft:2; 293ft_erastin_r:2; 293ft_rsl3_r:2; u2os:2; u2os_erastin_r:2; u2os_rsl3_r:2 |
| PRJNA996357 | 11 | 11 | no | manual_multi_class | csf_01:1; csf_02:1; csf_03:1; csf_07:1; csf_08:1; csf_09:1; csf_11:1; csf_12:1; csf_14:1; csf_16:1; csf_17:1 |

Summary: 48 BioProjects have curated sample sheets; 28 BioProjects have 2-4 classes and remain within the current DESeq2 class cap. Class distribution among the capped set: 2 classes: 10, 3 classes: 7, 4 classes: 11. Strict two-class designs with at least three samples per side: 8.

## Workflow

Each SRR entered the technical workflow once at the raw-read level and then branched into six comparable count-generation paths. FastQC was run on the raw reads to capture baseline sequence quality, read length, GC content, duplication, adapter content, and end-of-read quality decay. The same raw SRR was then processed as six alternatives:

| Branch | Cleaning step | Alignment step | Counting step |
|---|---|---|---|
| untrimmed | none | Bowtie2 | featureCounts |
| adapter_only | Trimmomatic adapter clipping | Bowtie2 | featureCounts |
| P5 | Trimmomatic adapter clipping plus Phred 5 quality trimming | Bowtie2 | featureCounts |
| P10 | Trimmomatic adapter clipping plus Phred 10 quality trimming | Bowtie2 | featureCounts |
| P20 | Trimmomatic adapter clipping plus Phred 20 quality trimming | Bowtie2 | featureCounts |
| P35 | Trimmomatic adapter clipping plus Phred 35 quality trimming | Bowtie2 | featureCounts |

In compact form, the workflow was:

```text
raw FASTQ
  |-- FastQC raw-read QC
  |
  |-- untrimmed ------------------------------> Bowtie2 -> sorted BAM -> featureCounts
  |-- Trimmomatic adapter_only ---------------> Bowtie2 -> sorted BAM -> featureCounts
  |-- Trimmomatic P5 -------------------------> Bowtie2 -> sorted BAM -> featureCounts
  |-- Trimmomatic P10 ------------------------> Bowtie2 -> sorted BAM -> featureCounts
  |-- Trimmomatic P20 ------------------------> Bowtie2 -> sorted BAM -> featureCounts
  `-- Trimmomatic P35 ------------------------> Bowtie2 -> sorted BAM -> featureCounts

six raw count vectors per SRR -> pairwise comparison to untrimmed counts
```

This design keeps the reference, annotation, aligner, counter, and SRR identity fixed while varying only the cleaning strategy. The primary comparison is therefore not whether a sample differs biologically from another sample, but whether the same sample produces a materially different gene-count vector after cleaning.

## Cleaning Strategies

For each SRR we compared the untrimmed count profile against cleaned profiles:

| Mode | Description | Non-default settings |
|---|---|---|
| untrimmed | No read cleaning before alignment | none |
| adapter_only | Adapter clipping only | `ILLUMINACLIP:<adapter_fasta>:2:30:10` |
| P5 | Adapter clipping plus Phred 5 quality trimming | `ILLUMINACLIP:<adapter_fasta>:2:30:10`, `SLIDINGWINDOW:4:5`, `LEADING:5`, `TRAILING:5` |
| P10 | Adapter clipping plus Phred 10 quality trimming | `ILLUMINACLIP:<adapter_fasta>:2:30:10`, `SLIDINGWINDOW:4:10`, `LEADING:10`, `TRAILING:10` |
| P20 | Adapter clipping plus Phred 20 quality trimming | `ILLUMINACLIP:<adapter_fasta>:2:30:10`, `SLIDINGWINDOW:4:20`, `LEADING:20`, `TRAILING:20` |
| P35 | Adapter clipping plus Phred 35 quality trimming | `ILLUMINACLIP:<adapter_fasta>:2:30:10`, `SLIDINGWINDOW:4:35`, `LEADING:35`, `TRAILING:35` |

P35 was retained as a stress test but is treated separately from the main interpretation because it removes a large fraction of reads and behaves as an intentionally aggressive trimming condition.

The adapter FASTA was a manually curated collection derived from the adapter sequences distributed with Trimmomatic. It was used consistently across all adapter-containing branches so that the only difference between P5/P10/P20/P35 was the Phred quality threshold.

No minimum-length filter was added beyond the explicit Trimmomatic operations listed above. For paired-end data, paired surviving reads were passed to alignment and unpaired survivors were discarded; for single-end data, the surviving single-end reads were passed directly to alignment.

## Reference Genome and Annotation

Alignment and counting used human GRCh38.p14 resources from NCBI RefSeq. The Bowtie2 index was built from the NCBI Datasets genome package for assembly accession `GCF_000001405.40`, corresponding to Genome Reference Consortium Human Build 38 patch release 14 (`GRCh38.p14`, synonym `hg38`). The local NCBI Datasets archive contained the FASTA file `GCF_000001405.40_GRCh38.p14_genomic.fna` and was downloaded from NCBI Datasets (`https://www.ncbi.nlm.nih.gov/datasets/`) as an assembly data package.

The count annotation used by featureCounts was the local file `GRCh38_annotation.gtf`. Its header identifies it as NCBI RefSeq annotation for the same assembly:

```text
genome-build: GRCh38.p14
genome-build-accession: NCBI_Assembly:GCF_000001405.40
annotation-source: NCBI RefSeq GCF_000001405.40-RS_2025_08
annotation-date: 08/01/2025
```

The genome FASTA bundled in the retained NCBI Datasets archive is from `GCF_000001405.40-RS_2024_08`; the featureCounts annotation file used for this analysis is the newer local RefSeq `RS_2025_08` GTF. Both use the same GRCh38.p14 assembly accession, so coordinate systems are matched, but the annotation release should be reported as `GCF_000001405.40-RS_2025_08` for the count tables.

| Resource | File used in workflow | Source/version | Role |
|---|---|---|---|
| Genome FASTA | `GCF_000001405.40_GRCh38.p14_genomic.fna` | NCBI Datasets, RefSeq assembly `GCF_000001405.40`, GRCh38.p14 / hg38 | Source sequence for Bowtie2 index |
| Bowtie2 index | `GRCh38_index.*.bt2` | Built from `GCF_000001405.40_GRCh38.p14_genomic.fna` | Alignment target |
| Gene annotation | `GRCh38_annotation.gtf` | NCBI RefSeq `GCF_000001405.40-RS_2025_08`, GTF 2.2 | featureCounts exon-to-gene counting |

For auditability, the main local annotation checksum was `454d94863eab453d323ac8320789c14c` for `GRCh38_annotation.gtf`. The Bowtie2 index component checksums were: `GRCh38_index.1.bt2` `fb02ade05f15fc6197a8680f921aa9ad`, `GRCh38_index.2.bt2` `5b7b95367abe6f6827ef0423333f3081`, `GRCh38_index.3.bt2` `8ae6e57438c650aa918cf5b98e9ecf21`, `GRCh38_index.4.bt2` `69c662344e6fa95fa9285213333c881a`, `GRCh38_index.rev.1.bt2` `4a6ab5538762f82903129e36543e0902`, and `GRCh38_index.rev.2.bt2` `b5fdbb02dfd74921a7f9cb7e72095118`.

## Alignment and Counting

Cleaned or untrimmed reads were aligned to the same reference index with Bowtie2. Paired-end libraries were aligned with `-1/-2`; single-end libraries with `-U`. The command used Bowtie2 defaults plus the declared Phred encoding, the appropriate layout option, and the requested thread count; no sensitivity preset such as `--very-sensitive` was added in the full technical matrix. The pipeline used 12 active threads unless otherwise scheduled.

Alignments were streamed through Samtools, converted to BAM, and coordinate-sorted. Gene-level counts were generated with featureCounts against the same GTF annotation for every cleaning mode. Counting used exon features and `gene_id` grouping. Paired-end libraries used featureCounts paired-end mode (`-p`). Strand-specific counting was not enabled, so featureCounts used its default unstranded mode.

A splice-aware STAR workflow was also piloted on a small subset of SRRs. Flattened STAR `ReadsPerGene.out.tab` outputs were retained for three pilot SRRs, and STAR timing records were available for eight SRRs. The observed STAR mapping wall times ranged from 424 to 14,741 seconds, with a median of 1,570.5 seconds. In the pilot timing files, STAR was not slower in wall-clock time than the Bowtie2-based branch processing; for example, in one large SRR the STAR mapping step took 3,474 seconds, while the complete Bowtie2-based six-branch technical matrix took 48,320 seconds after FastQC. These pilot runs confirmed that STAR was feasible for individual samples, but extending the technical design to a second splice-aware workflow would have required repeating the six cleaning/alignment/counting branches for the full SRR set using a substantially more memory-intensive aligner and larger scheduling footprint. We therefore used Bowtie2 for the full technical matrix and treat the analysis as an internally controlled preprocessing comparison rather than an optimized RNA-seq quantification workflow.

## Technical Metrics

For each SRR, raw gene-count vectors from the cleaned modes were compared against the untrimmed vector. The primary similarity metrics were Pearson correlation and Jensen-Shannon divergence. We summarized these per SRR, per BioProject, and globally, and generated paired plots with and without P35.

For each comparison, genes were matched by `gene_id` across the two featureCounts outputs for the same SRR. Pearson correlation measured linear similarity of the raw count vectors, while Jensen-Shannon divergence measured distributional divergence after converting the two raw count vectors into comparable gene-count proportions. The untrimmed branch was used as the reference profile for adapter-only, P5, P10, P20, and P35. We also summarized selected direct branch comparisons, such as P5 versus P35, to quantify the effect of increasingly aggressive quality thresholds.

FastQC outputs were parsed from `fastqc_data.txt` for read-level quality, length, GC, duplication, adapter, and end-of-read quality features. Paired-end libraries with separate FastQC reports were collapsed to an SRR-level summary for the quality analyses. We then ran sensitivity checks that balanced samples across quality bins and tail-quality bins to ask whether poorer sequence quality predicts larger count-profile disruption. Quality-balanced analyses divided eligible SRRs into bins with similar numbers of samples and then compared cleaning effects across those bins. Tail-quality analyses used the final part of each read, where quality decay is expected to appear first, and were stratified by read-length bucket so that short- and long-read libraries were not pooled into a single tail-quality scale.

Suspicious or poorly explainable samples identified during QC/outlier review were not deleted from the archive; instead, they were reported separately and excluded from the filtered/nonsuspicious sensitivity plots. This preserves the ability to inspect technical failures while preventing a small set of outliers from defining the main quality-dependence interpretation.



## Reproducible Data-selection Tables

The tables below are generated from the queue database and curated metadata files. They document the SRR and BioProject selection steps used to define the technical analysis set and the later DESeq2/GSEA follow-up panel.


In [1]:
from pathlib import Path
import sqlite3
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

ANALYSIS = Path("/pc2/users/o/omiks001/hpc-prf-omiks/ja/analysis")
TECH = ANALYSIS / "technical"
DB = Path("/pc2/users/o/omiks001/srr_queue.db")

with sqlite3.connect(DB) as con:
    queue = pd.read_sql_query(
        "SELECT project_id, srr_id, status, run_status FROM srr_queue",
        con,
    )

platform = pd.read_csv(TECH / "srr_platform_check.tsv", sep="\t")
platform = platform.rename(columns={
    "run_accession": "SRR_ID",
    "study_accession": "project_id",
    "instrument_platform": "platform",
})
deseq_summary = pd.read_csv(ANALYSIS / "deseq2_metadata" / "deseq2_metadata_summary.tsv", sep="\t")
deseq_summary["within_2_4_class_cap"] = deseq_summary["n_classes"].between(2, 4)

queue_summary = pd.DataFrame([
    {"selection_step": "Initial acquisition queue", "bioprojects": queue["project_id"].nunique(), "srrs": queue["srr_id"].nunique()},
    {"selection_step": "Completed queue entries", "bioprojects": queue.loc[queue["status"].eq("done"), "project_id"].nunique(), "srrs": queue.loc[queue["status"].eq("done"), "srr_id"].nunique()},
    {"selection_step": "Not completed", "bioprojects": "", "srrs": queue.loc[~queue["status"].eq("done"), "srr_id"].nunique()},
    {"selection_step": "Platform/library evaluated", "bioprojects": platform["project_id"].nunique(), "srrs": len(platform)},
    {"selection_step": "Illumina retained", "bioprojects": platform.loc[platform["is_illumina"].eq("yes"), "project_id"].nunique(), "srrs": int(platform["is_illumina"].eq("yes").sum())},
    {"selection_step": "Non-Illumina archived", "bioprojects": platform.loc[platform["is_illumina"].ne("yes"), "project_id"].nunique(), "srrs": int(platform["is_illumina"].ne("yes").sum())},
    {"selection_step": "Illumina RNA-seq, broad flag", "bioprojects": platform.loc[platform["is_illumina"].eq("yes") & platform["is_rnaseq"].eq("yes"), "project_id"].nunique(), "srrs": int((platform["is_illumina"].eq("yes") & platform["is_rnaseq"].eq("yes")).sum())},
    {"selection_step": "Illumina RNA-Seq strategy, strict curated panel", "bioprojects": deseq_summary["project_id"].nunique(), "srrs": int(deseq_summary["n_runs"].sum())},
    {"selection_step": "2-4 class cap for biological DESeq2 pass", "bioprojects": int(deseq_summary["within_2_4_class_cap"].sum()), "srrs": int(deseq_summary.loc[deseq_summary["within_2_4_class_cap"], "n_runs"].sum())},
])

queue_status_breakdown = (
    queue.groupby(["status", "run_status"], dropna=False)
    .agg(srrs=("srr_id", "nunique"), bioprojects=("project_id", "nunique"))
    .reset_index()
    .sort_values(["status", "run_status"])
)

platform_breakdown = (
    platform.groupby(["is_illumina", "is_rnaseq", "platform", "library_strategy"], dropna=False)
    .agg(srrs=("SRR_ID", "nunique"), bioprojects=("project_id", "nunique"))
    .reset_index()
    .sort_values(["is_illumina", "is_rnaseq", "srrs"], ascending=[True, True, False])
)

illumina_non_rnaseq_breakdown = (
    platform.loc[platform["is_illumina"].eq("yes") & platform["is_rnaseq"].ne("yes")]
    .groupby(["library_strategy", "library_source"], dropna=False)
    .agg(srrs=("SRR_ID", "nunique"), bioprojects=("project_id", "nunique"))
    .reset_index()
    .sort_values(["srrs", "library_strategy"], ascending=[False, True])
)

class_distribution = pd.DataFrame({
    "class_group": ["1 class", "2 classes", "3 classes", "4 classes", ">4 classes"],
    "bioprojects": [
        int(deseq_summary["n_classes"].eq(1).sum()),
        int(deseq_summary["n_classes"].eq(2).sum()),
        int(deseq_summary["n_classes"].eq(3).sum()),
        int(deseq_summary["n_classes"].eq(4).sum()),
        int(deseq_summary["n_classes"].gt(4).sum()),
    ],
})
class_distribution["included_in_2_4_class_cap"] = class_distribution["class_group"].isin(["2 classes", "3 classes", "4 classes"])
class_distribution.loc[len(class_distribution)] = ["Total curated BioProjects", int(class_distribution["bioprojects"].sum()), False]

capped_bioprojects = deseq_summary.loc[
    deseq_summary["within_2_4_class_cap"],
    ["project_id", "n_runs", "n_classes", "metadata_status", "condition_counts"],
].reset_index(drop=True)

display(queue_summary)
display(queue_status_breakdown)
display(platform_breakdown)
display(illumina_non_rnaseq_breakdown)
display(class_distribution)
display(capped_bioprojects)


,selection_step,bioprojects,srrs
0,Initial acquisition queue,55,1247
1,Completed queue entries,54,1201
2,Not completed,,46
3,Platform/library evaluated,54,1191
4,Illumina retained,51,1155
5,Non-Illumina archived,4,36
6,"Illumina RNA-seq, broad flag",49,917
7,"Illumina RNA-Seq strategy, strict curated panel",48,892
8,2-4 class cap for biological DESeq2 pass,28,485


,status,run_status,srrs,bioprojects
0,abandoned,COUNT_OUTPUTS_INCOMPLETE_ABANDONED,5,3
1,abandoned,FETCH_ABANDONED,11,6
2,abandoned,PROJECT_ABANDONED_LT70_RESULTS,20,1
3,aborted,SLURM_ABORTED,9,4
4,done,DONE,81,25
5,done,DONE_EARLY,5,3
6,done,DONE_NO_STAR,250,25
7,done,DONE_RESCUED,19,9
8,done,INITIALIZED,846,49
9,todo,FETCH_FAILED,1,1


,is_illumina,is_rnaseq,platform,library_strategy,srrs,bioprojects
0,no,no,OXFORD_NANOPORE,Synthetic-Long-Read,4,1
1,no,yes,BGISEQ,RNA-Seq,24,2
2,no,yes,OXFORD_NANOPORE,RNA-Seq,8,1
8,yes,no,ILLUMINA,Hi-C,58,2
11,yes,no,ILLUMINA,WGS,50,5
3,yes,no,ILLUMINA,AMPLICON,31,1
7,yes,no,ILLUMINA,ChIP-Seq,27,4
9,yes,no,ILLUMINA,OTHER,25,2
10,yes,no,ILLUMINA,POOLCLONE,16,1
5,yes,no,ILLUMINA,Bisulfite-Seq,15,2


,library_strategy,library_source,srrs,bioprojects
6,Hi-C,GENOMIC,58,2
10,WGS,GENOMIC,50,5
5,ChIP-Seq,GENOMIC,27,4
0,AMPLICON,TRANSCRIPTOMIC,21,1
7,OTHER,GENOMIC,19,1
9,POOLCLONE,GENOMIC,16,1
3,Bisulfite-Seq,GENOMIC,15,2
11,WXS,GENOMIC,14,2
1,AMPLICON,TRANSCRIPTOMIC SINGLE CELL,10,1
8,OTHER,METATRANSCRIPTOMIC,6,1


,class_group,bioprojects,included_in_2_4_class_cap
0,1 class,3,False
1,2 classes,10,True
2,3 classes,7,True
3,4 classes,11,True
4,>4 classes,17,False
5,Total curated BioProjects,48,False


,project_id,n_runs,n_classes,metadata_status,condition_counts
0,PRJNA1014965,37,2,manual_two_class,normal:20; tumor:17
1,PRJNA1120369,38,3,manual_multi_class,mega_01:14; mega_02:16; wt:8
2,PRJNA1127555,12,3,manual_multi_class,engager:4; naive:4; tumor:4
3,PRJNA1165739,40,4,manual_multi_class,hl60_ccl240:10; k562_ccl243:10; kasumi1_crl272...
4,PRJNA1175639,16,4,manual_multi_class,brown_differentiated:4; brown_preadipocyte_d0:...
5,PRJNA321028,16,3,manual_multi_class,stage_i_endometrial_cancer:4; stage_ii_endomet...
6,PRJNA321087,19,4,manual_multi_class,cancer:8; epithelium:5; gland:2; muscle:4
7,PRJNA321967,12,4,manual_multi_class,primary_spermatocytes:3; sertoli_cells:3; sper...
8,PRJNA352875,12,4,manual_multi_class,control:3; rho0:3; rho0_tnf:3; tnf_treated_con...
9,PRJNA373978,15,2,two_class_candidate,leptotene_zygotene_spermatocytes:5; pachytene_...


## Timing and Throughput Method

Timing and throughput accounting became important only later in the analysis, after a substantial part of the processing queue had already been run. Consequently, timing statistics are available for a subset of the processed data rather than for every SRR in the acquisition queue. The quantitative timing and throughput results are reported in `results.ipynb`; this section records only how those timings were captured.

Processing time was captured at the level of each pipeline stage rather than only as a total wall-clock duration. For each timed SRR and cleaning branch, the workflow recorded start and end times for major steps where available: SRA/FASTQ conversion, FastQC, Trimmomatic, Bowtie2 alignment, Samtools sorting, and featureCounts counting. Stage durations were stored together with the cleaning mode, SRR ID, BioProject ID, read counts, input size where available, and the number of requested processing cores.

From these records the results notebook derives per-stage wall-clock duration, core-seconds, reads per second, megabytes per second, and throughput per core. P35 timing is interpreted cautiously because aggressive trimming can make downstream alignment and counting appear artificially fast by removing many reads before those stages.


## Analysis Artifacts

All technical outputs were collected under `technical/`. The main reproducibility tables are `technical/global_summary.tsv` for per-SRR cleaning comparisons, `technical/per_srr_eval.tsv` for SRR-level count-profile metrics, `technical/per_project_summary.tsv` for BioProject-level summaries, `technical/trimmomatic_summary_by_mode.tsv` and `technical/trimmomatic_detail.tsv` for read-retention statistics, `technical/throughput_summary.tsv` and `technical/throughput_detail.tsv` for runtime summaries, `technical/quality_balanced_correlation_loss_summary.tsv` for quality-bin sampling, and `technical/tail_quality_correlation_loss_summary.tsv` for tail-quality/read-length sampling. The `technical/plots/` directory contains the corresponding visual summaries, and `technical/per_project/` contains per-BioProject count-evaluation tables.

## Limitations

Bowtie2 is not splice-aware, so the absolute read-to-gene assignment is not intended to replace a full RNA-seq differential-expression workflow built around a splice-aware aligner or transcript-aware quantifier. For the technical question asked here, however, this limitation is internally controlled: every cleaning branch of a given SRR used the same reference, annotation, aligner, counter, and counting rules. Therefore, branch-to-branch differences primarily measure the effect of cleaning on the resulting count profile rather than the absolute optimality of the RNA-seq quantification workflow.

The STAR pilot data should be interpreted as feasibility evidence, not as a formal benchmark showing that STAR is intrinsically slower than Bowtie2. STAR can be efficient for individual RNA-seq alignments; the limiting factor for a full STAR-based duplication of the six-branch cleaning matrix was the added memory, storage, and scheduler burden within the available allocation and study scope.

The technical analysis also uses raw count-vector similarity, not biological normalization or statistical differential-expression modeling. That is deliberate for this part of the study: the goal was to test whether preprocessing changes the count vectors themselves before asking the separate biological question of whether DESeq2 and pathway-level conclusions change.

## Software

| Tool | Version used/observed | Role |
|---|---|---|
| SRA Toolkit | 3.4.1 | SRR retrieval and FASTQ conversion |
| FastQC | 0.12.1 | Raw-read quality summaries |
| Trimmomatic | 0.40 | Adapter and quality trimming |
| Bowtie2 | 2.5.4 | Read alignment |
| STAR | 2.7.11b | Pilot splice-aware alignment |
| Samtools | 1.23 / htslib 1.23 | BAM conversion and sorting |
| featureCounts / Subread | 2.1.1 | Gene-level counting |
| Java/OpenJDK | 11.0.25 | Trimmomatic runtime |
| Python | 3.14.3 | Analysis table generation and plotting scripts |


The file paths and job wrappers are intentionally omitted here because they are cluster-specific. The reproducible methodological details are the tools, settings, retained sample definitions, and summary tables archived under `technical/`.
